# Lezione 01 — Che cos'è l'NLP

**Corso:** NLP Classico con Python · **Blocco 1:** Dal testo ai token
**Tempo stimato:** 90 minuti · **Capitolo del manuale:** `manual/01_que_es_el_nlp_IT.md`

## Cosa vedrai in questa lezione

- I **compiti tipici dell'NLP** e in quale lezione del corso si affronta ciascuno.
- La **pipeline classica**, dall'acquisizione dei testi alla valutazione, con una pipeline giocattolo che fallisce e viene corretta.
- **Perché il linguaggio è difficile**: ambiguità lessicale e sintattica (le altre difficoltà, nella lezione 2).
- La **ricca morfologia** dell'italiano e dello spagnolo: un esperimento che genera e conta le forme di un verbo, i clitici e genere e numero.
- Approccio **classico contro neurale**.
- Un esperimento guidato **«regole contro conteggi»** con un corpus piccolissimo in italiano e in spagnolo.

## Prerequisiti

- Python di base (funzioni, liste, dizionari) e nozioni di classificazione (corso di Machine Learning).
- Librerie: `spacy` (con `es_core_news_sm` e `it_core_news_sm`), `pandas`, `matplotlib`. La cella seguente le installa solo se mancano (funziona allo stesso modo in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. È la lezione più leggera del corso: poco codice e molti esempi. Tutti i testi sono brevi e scritti per il corso (*esempio illustrativo*). Ciò che qui viene solo mostrato come anteprima si studia con calma nella lezione indicata.

---
## Installazione e modelli

Questa cella installa ciò che manca e scarica i due modelli di spaCy. In un ambiente con tutto installato non fa nulla. In Colab la prima volta può richiedere un minuto.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("spacy", "spacy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")

import spacy

def cargar_modelo(nombre):
    """Carica un modello di spaCy e lo scarica se non è installato."""
    try:
        return spacy.load(nombre)
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre, "-q"], check=True)
        return spacy.load(nombre)

nlp_es = cargar_modelo("es_core_news_sm")
nlp_it = cargar_modelo("it_core_news_sm")
print("Modelli di spaCy pronti.")

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** dei generatori casuali. In questa lezione il caso interviene quasi mai, ma manteniamo l'abitudine: ogni studente deve ottenere gli stessi risultati.

In [ ]:
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)                          # seme fisso: tutti otteniamo gli stessi risultati
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
print("Librerie caricate correttamente.")

---
## 1. I compiti dell'NLP

Quasi ogni problema di NLP si può scrivere come **ingresso → uscita**. Ecco le sei famiglie di compiti del corso con un esempio reale in ciascuna lingua e la lezione in cui si lavora. La **generazione moderna** e la **traduzione automatica** sono fuori dal corso: i corsi di IA generativa sono il posto dove proseguire.

In [ ]:
# datos-es
EJEMPLOS = {
    "clf": ("Mi paquete no ha llegado y nadie responde", "Il mio pacco non è arrivato e nessuno risponde"),
    "ext": ("El 3 de marzo pagamos 1.250 euros a Talleres Ruiz", "Il 3 marzo abbiamo pagato 1.250 euro all'officina Rossi"),
    "s2s": ("La reunión se ha aplazado al jueves", "La riunione è stata rinviata a giovedì"),
    "gen": ("Estimado cliente, lamentamos el retraso de su pedido y…", "Gentile cliente, ci scusiamo per il ritardo del suo ordine e…"),
    "qa": ("¿Cuál es el plazo para devolver un producto?", "Qual è il termine per restituire un prodotto?"),
    "ling": ("Los niños comieron manzanas rojas", "I bambini hanno mangiato mele rosse"),
}

In [ ]:
TAREAS = {
    "clf":  ("Classificazione", "un'etichetta (categoria di ticket, sentimento)", "lezioni 5 e 6"),
    "ext":  ("Estrazione", "dati concreti (data, importo, organizzazione)", "lezione 7 (e regex nella 2)"),
    "s2s":  ("Sequenza-a-sequenza", "un altro testo (traduzione o riassunto)", "lezione 11 (riassunto); traduzione: fuori dal corso"),
    "gen":  ("Generazione", "testo nuovo a partire da un inizio", "lezione 8 (idea con n-grammi); moderna: fuori dal corso"),
    "qa":   ("Recupero e risposta", "i documenti che contengono la risposta", "lezione 11"),
    "ling": ("Analisi linguistica", "categorie grammaticali e sintassi", "lezioni 3 e 7"),
}

filas = []
for clave, (tarea, salida, donde) in TAREAS.items():
    ej_es, ej_it = EJEMPLOS[clave]
    filas.append({"compito": tarea, "esempio (spagnolo)": ej_es, "esempio (italiano)": ej_it,
                  "cosa esce": salida, "dove": donde})
tabla_tareas = pd.DataFrame(filas)
print(tabla_tareas.to_string(index=False))

**Cosa osservi?** Ogni riga si legge come «entra un testo, esce X». Nota che lo stesso compito esiste identico nelle due lingue, ma gli strumenti (liste di stopword, modelli di spaCy, lessici) sono diversi per ciascuna. E due delle sei righe (traduzione e generazione moderna) non si studiano in questo corso: sapere dove finisce ciò che è coperto fa parte della mappa.

---
## 2. La pipeline classica

Sei passi: **acquisizione → pulizia e normalizzazione → tokenizzazione → rappresentazione → modello → valutazione e uso**. Ciascuno ha la sua lezione nel corso (2 e 3 per i passi 2 e 3; 4 e 10 per la rappresentazione; 5, 6, 8 e 9 per il modello; 5, 6, 7, 11 e 12 per valutare e usare). I passi **si scelgono in funzione del compito** e **raramente si percorrono una sola volta**: valutando si torna indietro.

Ecco una pipeline giocattolo che decide se un ticket è urgente. Ogni passo è una funzione di una riga; **la tokenizzazione con `split()` è volutamente ingenua** e la correggeremo nella lezione 2.

In [ ]:
# datos-es
TICKETS = {"spa": "Mi pedido no ha llegado y necesito ayuda urgente.", "ita": "Il mio ordine non è arrivato e ho bisogno di aiuto urgente."}
PALABRAS_URGENTES = {"urgente", "urgentemente", "inmediatamente", "subito", "immediatamente"}

In [ ]:
def adquirir(idioma):
    return TICKETS[idioma]                                   # passo 1: qui, un testo già scritto

def limpiar(texto):
    return re.sub(r"\s+", " ", texto).strip().lower()         # passo 2

def tokenizar_ingenuo(texto):
    return texto.split()                                     # passo 3 (ingenuo)

def representar(tokens):
    return Counter(tokens)                                   # passo 4: un sacco di parole

def modelo(bolsa):
    return "urgente" if any(p in bolsa for p in PALABRAS_URGENTES) else "normal"   # passo 5: una regola

for idioma in ("spa", "ita"):
    texto = adquirir(idioma)
    tokens = tokenizar_ingenuo(limpiar(texto))
    print(idioma, len(tokens), tokens[-3:], "→", modelo(representar(tokens)))

**Cosa osservi?** L'ultimo token è `urgente.` (con il punto attaccato) e la regola non lo riconosce: il ticket esce «normal» senza alcun errore visibile. Il guasto è nel passo 3, non nel modello. Torniamo indietro e cambiamo solo quel passo.

In [ ]:
def tokenizar_regex(texto):
    return re.findall(r"\w+", texto)                         # il punto non resta più attaccato

for idioma in ("spa", "ita"):
    texto = adquirir(idioma)
    tokens = tokenizar_regex(limpiar(texto))
    print(idioma, len(tokens), tokens[-3:], "→", modelo(representar(tokens)))

Con la modifica, i due ticket escono «urgente». Ecco cosa significa che la pipeline **non è lineare**: un errore scoperto alla fine obbliga a tornare a un passo precedente. Nella lezione 2 vedrai cos'altro fallisce con `split()` e con `\w+` (apostrofi dell'italiano, cifre, contrazioni).

---
## 3. Perché il linguaggio è difficile

### 3.1 Ambiguità lessicale

Una stessa stringa ha più significati. Per un sacco di parole (lezione 4) le due occorrenze di `banco` sono **lo stesso token**.

In [ ]:
LEXICA = [
    ("banco", "Me senté en el banco del parque", "sedile"),          # dato-es
    ("banco", "Ingresé el dinero en el banco", "istituto finanziario"),          # dato-es
    ("vino", "Con la cena tomamos vino tinto", "bevanda"),          # dato-es
    ("vino", "Ayer vino mi hermana a verme", "forma del verbo venir"),          # dato-es
    ("pesca", "Ho mangiato una pesca matura", "frutto"),          # dato-es
    ("pesca", "La pesca è vietata nel lago", "attività"),          # dato-es
    ("calcio", "Il calcio è il suo sport preferito", "sport"),          # dato-es
    ("calcio", "Il latte contiene calcio", "elemento chimico"),          # dato-es
]

for palabra, frase, sentido in LEXICA:
    print(f"{palabra:>6} | {sentido:<22} | {frase}")

fr_a, fr_b = LEXICA[0][1], LEXICA[1][1]
print("Il token 'banco' è identico nelle due frasi?", "banco" in fr_a.split() and "banco" in fr_b.split())

**Cosa osservi?** Il programma vede esattamente la stessa stringa `banco` nelle due frasi; solo le parole intorno le distinguono. Per questo gli n-grammi (lezione 8) e gli *embedding* (lezione 10) guardano il contesto.

### 3.2 Ambiguità sintattica

`Vi al hombre con el telescopio` / `Ho visto l'uomo con il cannocchiale`: due letture, a seconda della parola a cui si collega il gruppo «con il cannocchiale». Come **anteprima** della lezione 7, mostriamo i componenti della *pipeline* di spaCy e a quale parola il parser collega quel gruppo.

In [ ]:
# datos-es
FRASES_SINTAXIS = [(nlp_es, "Vi al hombre con el telescopio", "telescopio"), (nlp_it, "Ho visto l'uomo con il cannocchiale", "cannocchiale")]

In [ ]:
for nlp, frase, palabra_clave in FRASES_SINTAXIS:
    doc = nlp(frase)
    objetivo = [t for t in doc if t.text == palabra_clave][0]
    print(nlp.meta["lang"], "|", nlp.pipe_names)
    print("   il gruppo si collega a:", objetivo.head.text, "(", objetivo.dep_, ")")

**Cosa osservi?** La *pipeline* ha diversi componenti (`tok2vec`, `morphologizer`, `parser`, `lemmatizer`, `ner`…) e il parser **sceglie** una lettura. Ricorda: **le due letture sono valide**; il modello ne sceglie una perché ha visto molte frasi simili, non perché «capisca» la scena. Per questo questa cifra non viene asserita in nessun test del corso: con un'altra versione del modello può cambiare.

> **Nota:** le altre difficoltà (ambiguità referenziale, contesto e pragmatica, variazione del testo reale) si studiano nella lezione 2, dove inizia il lavoro con il testo grezzo.

---
## 4. La ricca morfologia dell'italiano e dello spagnolo

### 4.1 Esperimento guidato: quante forme ha un verbo?

Generiamo con il codice i **tempi semplici regolari** del verbo `cantar` (spagnolo) e `cantare` (italiano) per le sei persone, più le forme non personali (infinito, gerundio e participio in quattro forme), e contiamo le **forme distinte**. Poi le confrontiamo con le cinque forme di un verbo inglese (`sing`, `sings`, `sang`, `sung`, `singing`).

In [ ]:
# datos-es
TERMINACIONES_ES = {
    "presente": ["o", "as", "a", "amos", "áis", "an"],
    "imperfecto": ["aba", "abas", "aba", "ábamos", "abais", "aban"],
    "indefinido": ["é", "aste", "ó", "amos", "asteis", "aron"],
    "futuro": ["aré", "arás", "ará", "aremos", "aréis", "arán"],
    "condicional": ["aría", "arías", "aría", "aríamos", "aríais", "arían"],
    "subjuntivo": ["e", "es", "e", "emos", "éis", "en"],
}
TERMINACIONES_IT = {
    "presente": ["o", "i", "a", "iamo", "ate", "ano"],
    "imperfetto": ["avo", "avi", "ava", "avamo", "avate", "avano"],
    "remoto": ["ai", "asti", "ò", "ammo", "aste", "arono"],
    "futuro": ["erò", "erai", "erà", "eremo", "erete", "eranno"],
    "condizionale": ["erei", "eresti", "erebbe", "eremmo", "ereste", "erebbero"],
    "congiuntivo": ["i", "i", "i", "iamo", "iate", "ino"],
}
NO_PERSONALES_ES = ["cantar", "cantando", "cantado", "cantada", "cantados", "cantadas"]
NO_PERSONALES_IT = ["cantare", "cantando", "cantato", "cantata", "cantati", "cantate"]
FORMAS_INGLES = ["sing", "sings", "sang", "sung", "singing"]

In [ ]:
def conjugar(raiz, terminaciones):
    """Restituisce {tempo: [6 forme]} aggiungendo ogni terminazione alla radice."""
    return {tiempo: [raiz + t for t in lista] for tiempo, lista in terminaciones.items()}

formas_es = conjugar("cant", TERMINACIONES_ES)
formas_it = conjugar("cant", TERMINACIONES_IT)

for nombre, formas in (("Spagnolo", formas_es), ("Italiano", formas_it)):
    print(nombre)
    for tiempo, lista in formas.items():
        print(f"  {tiempo:<13}", ", ".join(lista))

In [ ]:
def contar_formas(formas_personales, no_personales):
    lista = [f for lista in formas_personales.values() for f in lista] + no_personales
    return len(lista), len(set(lista)), lista

huecos_es, distintas_es, lista_es = contar_formas(formas_es, NO_PERSONALES_ES)
huecos_it, distintas_it, lista_it = contar_formas(formas_it, NO_PERSONALES_IT)

print(f"Spagnolo: {huecos_es} posti, {distintas_es} forme distinte")
print(f"Italiano: {huecos_it} posti, {distintas_it} forme distinte")
print(f"Inglese (sing): {len(FORMAS_INGLES)} forme distinte")
print(f"Volte in più di forme rispetto all'inglese: {distintas_es / len(FORMAS_INGLES):.1f} (spa) y {distintas_it / len(FORMAS_INGLES):.1f} (ita)")

repetidas_es = sorted(f for f, n in Counter(lista_es).items() if n > 1)
repetidas_it = sorted(f for f, n in Counter(lista_it).items() if n > 1)
print("Forme che coincidono in più di un posto (spa):", repetidas_es)
print("Forme che coincidono in più di un posto (ita):", repetidas_it)

In [ ]:
etiquetas = ["spagnolo", "italiano", "inglese"]
valores = [distintas_es, distintas_it, len(FORMAS_INGLES)]
plt.bar(etiquetas, valores, color=["#c0392b", "#27ae60", "#2980b9"])
for i, v in enumerate(valores):
    plt.text(i, v + 0.5, str(v), ha="center")
plt.title("Forme distinte di un verbo regolare (esempio illustrativo)")
plt.ylabel("Forme distinte")
plt.show()

**Cosa osservi?** Un verbo regolare in -ar / -are si scrive in **38** (spagnolo) e **37** (italiano) forme distinte con questi tempi, contro le **5** del verbo inglese: tra sette e otto volte di più. Il numero di forme distinte è inferiore a quello dei posti (42) perché alcune coincidono: `cantamos` è al tempo stesso presente e indefinido; in italiano `canti` vale per tre persone del congiuntivo e una del presente. E sono solo i tempi semplici di un verbo **regolare**. Per un sacco di parole ogni forma è una caratteristica distinta: con pochi dati questa dispersione fa male.

Ora un esperimento **aperto**: passiamo ogni forma isolata dal lemmatizzatore di spaCy e contiamo quante tornano all'infinito. Il risultato viene solo mostrato, non asserito.

In [ ]:
for nombre, nlp, lista, infinitivo in (("Spagnolo", nlp_es, lista_es, "cantar"), ("Italiano", nlp_it, lista_it, "cantare")):
    unicas = sorted(set(lista))
    lemas = {f: nlp(f)[0].lemma_ for f in unicas}
    vuelven = [f for f, lema in lemas.items() if lema == infinitivo]
    fallan = [(f, lemas[f]) for f in unicas if lemas[f] != infinitivo][:6]
    print(f"{nombre}: {len(vuelven)} su {len(unicas)} forme tornano all'infinito")
    print("   Esempi che non tornano:", fallan)

**Cosa osservi?** Il lemmatizzatore (con spaCy 3.8 e i modelli 3.8.0; può variare leggermente con altre versioni) restituisce l'infinito solo per una parte delle forme, e a volte inventa lemmi (`cantabai`). Una forma isolata non dà contesto: il lemmatizzatore dipende dalla categoria grammaticale, che dipende dalla frase. Lo vedremo nella lezione 3.

### 4.2 Clitici

I pronomi atoni si attaccano al verbo: `dámelo`, `dammelo`, `dándoselo`, `dandoglielo`. Una sola stringa contiene un verbo e due pronomi.

In [ ]:
# datos-es
CLITICOS = [(nlp_es, "Dámelo ahora"), (nlp_es, "Quiero dárselo mañana"), (nlp_it, "Dammelo adesso"), (nlp_it, "Voglio darglielo domani")]

In [ ]:
for nlp, frase in CLITICOS:
    doc = nlp(frase)
    print(f"{frase!r}: split() → {len(frase.split())} parole;", "spaCy →", [(t.text, t.lemma_) for t in doc])

**Cosa osservi?** Né `split()` né il tokenizzatore di spaCy separano `dámelo` in `da` + `me` + `lo`, e il lemma che il modello restituisce per le forme con clitico non è il verbo (con questi modelli; può variare con altre versioni). È un problema che compare nelle lezioni 2 e 3.

### 4.3 Genere e numero

Gli aggettivi concordano con il sostantivo: quattro forme in spagnolo e in italiano, una in inglese.

In [ ]:
adjetivos = {"spagnolo": ["rojo", "roja", "rojos", "rojas"], "italiano": ["rosso", "rossa", "rossi", "rosse"], "inglese": ["red"]}          # dato-es
for idioma, formas in adjetivos.items():
    print(f"{idioma:<9} {len(set(formas))} forma/e: {formas}")

for nlp, frase in ((nlp_es, "Los coches rojos son bonitos"), (nlp_it, "Le macchine rosse sono belle")):          # dato-es
    print([(t.text, t.lemma_) for t in nlp(frase)])

**Cosa osservi?** In una frase completa il lemmatizzatore restituisce il lemma maschile singolare (`rojo`, `rosso`, `bonito`, `bello`): con il contesto funziona molto meglio che con forme isolate. Va comunque controllato in ogni lingua. **Regola pratica:** gli strumenti pensati per l'inglese non si trasferiscono senza adattamento (stopword, lemmatizzatori, accenti, apostrofi).

---
## 5. Approccio classico contro approccio neurale

Nessuno dei due è sempre migliore. Questa tabella è onesta su quando conviene ciascuno; nella **lezione 12** li confronterai con delle cifre su uno stesso problema.

| Criterio | Classico | Neurale (transformer) |
|---|---|---|
| Dati necessari | Pochi: spesso bastano centinaia di esempi | Il preaddestrato sa molto; adattarlo bene richiede più esempi |
| Costo di calcolo | Basso (un portatile, secondi) | Alto (memoria, tempo, spesso GPU) |
| Spiegabilità | Alta: si leggono regole e pesi | Bassa: è difficile dire perché ha scelto un'etichetta |
| Robustezza (variazione, contesto) | Fragile con ortografia, negazione, ironia | Molto migliore con contesto e variazione |
| Lingue con poche risorse | Dipende da risorse per lingua | Il multilingue aiuta, con distorsione verso le lingue grandi |
| Manutenzione | Semplice da correggere e aggiornare | Più costosa (versioni, rilascio, cambi di comportamento) |

**Conviene il classico** quando ci sono pochi dati etichettati, serve spiegare ogni decisione, il budget di calcolo è minimo o un'espressione regolare risolve già il problema. **Conviene il neurale** quando contesto e variazione pesano molto e ci sono risorse per usarlo.

---
## 6. Esperimento guidato: regole contro conteggi

Un mini-problema di **sentimento** con un corpus piccolissimo scritto per il corso (**esempio illustrativo**): 12 frasi di addestramento e 8 di test per lingua, bilanciate (6 + 6 per addestrare, 4 + 4 per provare) e con **due casi di negazione** in ciascun insieme di test. Solo Python puro, senza scikit-learn.

Tre classificatori: (1) **regole**: liste di parole positive e negative; (2) **regole con negazione**: un negatore inverte la polarità della parola che compare nelle tre posizioni successive; (3) **conteggi**: un `Counter` di parole per classe, contato nell'addestramento. In caso di parità o se nessuna parola è nota, il classificatore **non decide** (`None`) e conta come errore.

> **Attenzione:** con 8 frasi di test, una sola risposta giusta di differenza sposta di 12,5 punti. Le percentuali sono **illustrative, non generalizzabili**.

In [ ]:
# datos-es
ENTRENAMIENTO_ES = [
    ("el producto es excelente y muy útil", "pos"),
    ("me encantó el servicio, todo perfecto", "pos"),
    ("la entrega fue rápida y el envío llegó bien", "pos"),
    ("muy buena calidad, estoy contento con la compra", "pos"),
    ("atención amable y precio justo", "pos"),
    ("una experiencia fantástica, lo recomiendo", "pos"),
    ("el producto es malo y muy caro", "neg"),
    ("odié el servicio, todo un desastre", "neg"),
    ("la entrega fue lenta y el envío llegó roto", "neg"),
    ("pésima calidad, estoy decepcionado con la compra", "neg"),
    ("atención horrible y precio abusivo", "neg"),
    ("una experiencia terrible, lo desaconsejo", "neg"),
]
PRUEBA_ES = [
    ("un producto excelente, todo perfecto", "pos"),
    ("la atención fue amable y el envío rápido", "pos"),
    ("el envío no fue lento", "pos"),
    ("una compra genial", "pos"),
    ("un desastre, todo muy caro", "neg"),
    ("la calidad no es buena", "neg"),
    ("una compra lamentable", "neg"),
    ("¡qué maravilla, se rompió al primer día!", "neg"),
]
ENTRENAMIENTO_IT = [
    ("il prodotto è eccellente e molto utile", "pos"),
    ("mi è piaciuto il servizio, tutto perfetto", "pos"),
    ("la consegna è stata veloce e la spedizione è arrivata bene", "pos"),
    ("ottima qualità, sono contento dell'acquisto", "pos"),
    ("assistenza gentile e prezzo giusto", "pos"),
    ("un'esperienza fantastica, lo consiglio", "pos"),
    ("il prodotto è pessimo e molto caro", "neg"),
    ("ho odiato il servizio, un vero disastro", "neg"),
    ("la consegna è stata lenta e la spedizione è arrivata rotta", "neg"),
    ("scarsa qualità, sono deluso dell'acquisto", "neg"),
    ("assistenza orribile e prezzo esagerato", "neg"),
    ("un'esperienza terribile, lo sconsiglio", "neg"),
]
PRUEBA_IT = [
    ("un prodotto eccellente, tutto perfetto", "pos"),
    ("l'assistenza è stata gentile e la spedizione veloce", "pos"),
    ("la spedizione non è stata lenta", "pos"),
    ("un acquisto geniale", "pos"),
    ("un disastro, tutto molto caro", "neg"),
    ("la qualità non è buona", "neg"),
    ("un acquisto scadente", "neg"),
    ("che meraviglia, si è rotto il primo giorno!", "neg"),
]
POSITIVAS_ES = {"excelente", "útil", "encantó", "perfecto", "rápida", "rápido", "bien", "buena", "bueno",
                "contento", "amable", "justo", "fantástica", "recomiendo", "genial", "maravilla", "gustó"}
NEGATIVAS_ES = {"malo", "caro", "odié", "desastre", "lenta", "lento", "roto", "pésima", "decepcionado",
                "horrible", "abusivo", "terrible", "desaconsejo", "lamentable"}
POSITIVAS_IT = {"eccellente", "utile", "piaciuto", "perfetto", "veloce", "bene", "buona", "buono", "ottima",
                "contento", "gentile", "giusto", "fantastica", "consiglio", "geniale", "meraviglia"}
NEGATIVAS_IT = {"pessimo", "caro", "odiato", "disastro", "lenta", "lento", "rotta", "rotto", "scarsa", "deluso",
                "orribile", "esagerato", "terribile", "sconsiglio", "scadente"}
NEGADORES_ES = {"no", "nunca", "ni", "tampoco"}
NEGADORES_IT = {"non", "mai", "né"}

In [ ]:
def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())

def clasificar_reglas(texto, positivas, negativas, negadores=None, ventana=3):
    palabras = tokenizar(texto)
    puntos_pos = puntos_neg = 0
    for i, palabra in enumerate(palabras):
        valor = 1 if palabra in positivas else -1 if palabra in negativas else 0
        if valor and negadores and any(x in negadores for x in palabras[max(0, i - ventana):i]):
            valor = -valor                       # «non è buono»: la negazione inverte la polarità
        puntos_pos += valor > 0
        puntos_neg += valor < 0
    return "pos" if puntos_pos > puntos_neg else "neg" if puntos_neg > puntos_pos else None

def entrenar_conteos(ejemplos):
    conteos = {"pos": Counter(), "neg": Counter()}
    for texto, etiqueta in ejemplos:
        conteos[etiqueta].update(tokenizar(texto))
    return conteos

def clasificar_conteos(texto, conteos):
    peso_pos = sum(conteos["pos"][p] for p in tokenizar(texto))
    peso_neg = sum(conteos["neg"][p] for p in tokenizar(texto))
    return "pos" if peso_pos > peso_neg else "neg" if peso_neg > peso_pos else None

def exactitud(clasificador, ejemplos):
    return sum(clasificador(texto) == real for texto, real in ejemplos) / len(ejemplos)

DATOS = {
    "spa": (ENTRENAMIENTO_ES, PRUEBA_ES, POSITIVAS_ES, NEGATIVAS_ES, NEGADORES_ES),
    "ita": (ENTRENAMIENTO_IT, PRUEBA_IT, POSITIVAS_IT, NEGATIVAS_IT, NEGADORES_IT),
}
print("Bilanciamento dell'addestramento:", {i: Counter(e for _, e in d[0]) for i, d in DATOS.items()})
print("Bilanciamento del test:", {i: Counter(e for _, e in d[1]) for i, d in DATOS.items()})

In [ ]:
clasificadores = {}
resultados = {}
for idioma, (entren, prueba, pos, neg, nega) in DATOS.items():
    conteos = entrenar_conteos(entren)
    clasificadores[idioma] = {
        "regole": lambda t, pos=pos, neg=neg: clasificar_reglas(t, pos, neg),
        "regole + negazione": lambda t, pos=pos, neg=neg, nega=nega: clasificar_reglas(t, pos, neg, nega),
        "conteggi": lambda t, conteos=conteos: clasificar_conteos(t, conteos),
    }
    resultados[idioma] = {nombre: exactitud(f, prueba) for nombre, f in clasificadores[idioma].items()}

tabla = pd.DataFrame(resultados).rename(columns={"spa": "spagnolo", "ita": "italiano"})
print((tabla * 100).round(1).astype(str) + " %")

In [ ]:
nombres = list(resultados["spa"].keys())
ancho = 0.35
posiciones = np.arange(len(nombres))
plt.bar(posiciones - ancho / 2, [resultados["spa"][n] for n in nombres], ancho, label="spagnolo", color="#c0392b")
plt.bar(posiciones + ancho / 2, [resultados["ita"][n] for n in nombres], ancho, label="italiano", color="#27ae60")
plt.xticks(posiciones, nombres)
plt.ylim(0, 1)
plt.ylabel("Accuratezza su 8 frasi")
plt.title("Regole contro conteggi (esempio illustrativo, 8 frasi di test)")
plt.legend()
plt.show()

**Cosa osservi?** Prima di guardare le percentuali, guarda **come fallisce** ciascun classificatore. Vediamo gli errori frase per frase.

In [ ]:
for idioma, (entren, prueba, pos, neg, nega) in DATOS.items():
    print("=" * 8, idioma)
    for nombre, f in clasificadores[idioma].items():
        errores = [(texto, real, f(texto) or "—") for texto, real in prueba if f(texto) != real]
        print(f"Classificatore '{nombre}': {len(errores)} errori")
        for texto, real, predicho in errores:
            print(f"    {texto!r}  reale={real}  predetto={predicho}")

In [ ]:
for idioma, (entren, prueba, pos, neg, nega) in DATOS.items():
    vocabulario = {p for texto, _ in entren for p in tokenizar(texto)}
    palabras_prueba = [p for texto, _ in prueba for p in tokenizar(texto)]
    fuera = [p for p in palabras_prueba if p not in vocabulario]
    print(f"{idioma}: {len(fuera)} su {len(palabras_prueba)} parole di test non compaiono nell'addestramento")

**Cosa osservi?**

- Le **regole senza negazione** falliscono dove c'è un `no`/`non` davanti a una parola con carica (`el envío no fue lento`, `la qualità non è buona`); con la negazione questi casi si correggono.
- I **conteggi** falliscono per **parole mai viste** (`genial`, `lamentable`, `geniale`, `scadente`): con sole 12 frasi di addestramento, buona parte del vocabolario di test è nuova. Falliscono anche con la negazione, perché `no` e `lento` non sono collegati in un sacco di parole.
- **Nessuno** dei tre capisce l'ironia di `¡qué maravilla, se rompió al primer día!`.
- Un solo numero di accuratezza nasconde tutto questo. Per questo guardiamo gli errori, e per questo il 62,5 % o l'87,5 % di qui sono **illustrativi**: non si possono generalizzare ad altri dati. Nella lezione 5 imparerai a valutare con dati sufficienti e con la metrica adeguata.

---
## 7. Riepilogo e passo successivo

- L'NLP trasforma il testo in dati: quasi ogni compito si scrive come **ingresso → uscita** (classificare, estrarre, trasformare, generare, recuperare, analizzare).
- La **pipeline classica** (acquisizione → pulizia → tokenizzazione → rappresentazione → modello → valutazione) si sceglie in base al compito e raramente si percorre una sola volta.
- Il linguaggio è difficile per l'**ambiguità** (qui, lessicale e sintattica); il modello «sceglie», non «capisce». Riferimento, contesto e variazione si studiano nella lezione 2.
- L'italiano e lo spagnolo hanno una **ricca morfologia**: 38 e 37 forme distinte per un verbo regolare contro le 5 dell'inglese, clitici e concordanza di genere e numero. Gli strumenti si adattano lingua per lingua.
- **Classico e neurale** si confrontano con dei criteri (dati, costo, spiegabilità, robustezza, risorse, manutenzione); nella lezione 12 si confrontano con delle cifre.
- Una percentuale su 8 frasi è solo **illustrativa**; bisogna guardare gli errori.

**Prossimo passo:** nella lezione 2, *Testo grezzo e tokenizzazione*, correggerai davvero il passo 3 della pipeline: Unicode, espressioni regolari e tokenizzazione. Prima, fai il notebook di pratica di questa lezione.